# nano Beatrix — what one head's channels carried, and where the path head looks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_pillar_room.ipynb)

**The question**: In the path stage, head 7 of every block gave its eight address pairs to the path since the text's last choice point. On plain text those pairs carry no record address, so in the plain-rows models they are eight position-free content channels of that head. The byte-after path models lean on the path (switching it off costs 0.10-0.13 bits a byte) yet read held-out text no better than the plain-rows models on average: the lookup pays where a path repeats, and the given-up channels cost elsewhere. Before more of a head is given to the path, this notebook reads two things from the stored models, training nothing:

| read | on | what |
|---|---|---|
| the channels | the six plain-rows models | each head's eight anchor pairs removed from its scores in every block (one head at a time), and head 7's relative band, content pairs, all sixteen pairs and its output removed: the held-out text's cost of each, at the positions whose path occurred earlier in the row and elsewhere |
| the channels | the twelve path models | head 7's anchor pairs (the path's channel) and its output removed |
| the look | the six plain-rows and six byte-after models | every head's attention, at the positions whose path occurred earlier in the row, on the bytes that followed the earlier copies of the path, against the share a uniform attention would give, by the path's length |

A removal zeroes the head's query and key on the named pairs after the block's own rotation, so those pairs add nothing to the head's scores; the output removal zeroes the head's values. The rows are the text read's: the trainer's 128 held-out documents, 4,096 bytes each; each model's own bits are checked against its stored text read first.

**The cells**: C1 pins the trainer and holds the decision block; C2 writes the held-out documents and reads the eighteen models in turn (resume-first: a record on the Hub is not read again), shipping each record to [AbstractPhil/geolip-rotary-experiments](https://huggingface.co/AbstractPhil/geolip-rotary-experiments) `bed/reports/room/` as it lands; C3 renders the tables.

**Before C1**: Runtime > Change runtime type > the G4 card. The notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on, for the records' upload (the models are read without a token). Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` starts no new model. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/room_read.py`, `tests/test_room_read.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK (one G4 runtime). RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.93" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 93), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the held-out documents, a model's weights while it is read, the records
REPO = 'AbstractPhil/geolip-rotary-experiments'      # the stored models and their text reads; the records ship to bed/reports/room/
ROOM_DIR = 'bed/reports/room'
TAIL = '_L24000_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_p75d_gag50_cp4_gc48_kh50_va1_pt10b4h0_krt16r218_'
TAG_PLAIN = lambda s: f'alhena-both_s{s}{TAIL}ahdigit1_plain16'                  # the plain-rows models (the comparison of the path stage)
TAG_PATH = lambda f, s: f'alhena-both_s{s}{TAIL}pb{f[0]}8h7_ahdigit1_pillar1'    # the path models: 'next' (the byte after; _pbn8h7) or 'same' (_pbs8h7)
LOOK_ROWS = 16                                         # the look on the first 16 held-out rows
CORES = ([(f'plain s{s}', TAG_PLAIN(s), 'plain', LOOK_ROWS) for s in range(1, 7)]
         + [(f'byte-after s{s}', TAG_PATH('next', s), 'path', LOOK_ROWS) for s in range(1, 7)]
         + [(f'same-path s{s}', TAG_PATH('same', s), 'path', 0) for s in range(1, 7)])
TEXT_DOCS, TEXT_MB = 128, 16                           # the text read's rows: the trainer's 128 held-out documents in its batches of 16
BASE_TOL = 1e-4                                        # a model's own bits here against its stored text read
PLAN_MIN = 20.0                                        # the planned minutes for the eighteen models on this card
print('decision block:', len(CORES), 'models', [c[0] for c in CORES], '| the look on', LOOK_ROWS, 'rows | the text on', TEXT_DOCS, 'documents')

In [ ]:
# C2 — THE READS: the held-out documents written alone, then each model in turn (job size 18; the time spent and left on every line): its
# weights and its stored text read fetched without a token, the read (its own held-out bits; each removal; the look), its own bits checked
# against the stored text read, the record shipped to bed/reports/room/ as it lands (a background shipper: the card never waits on the
# Hub). Resume-first: a record on the Hub or in OUT/room/ is not read again.
import json, shutil, time
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from geolip.nano_beatrix import bed_corpus
from geolip.nano_beatrix import room_read as rr
from geolip.nano_beatrix import rotary_scale as rs
os.makedirs(f'{OUT}/bed', exist_ok=True)
HELD = bed_corpus.write_heldout(f'{OUT}/bed')
_hub = set(HfApi().list_repo_files(REPO))
_missing = [t for _, t, _, _ in CORES if f'bed/weights/{t}/model.safetensors' not in _hub or f'bed/reports/text/{rs.text_name(t)}.json' not in _hub]
assert not _missing, ('a model or its text read is not on the Hub', _missing[:2])
ship = rs.Shipper(REPO) if HF_TOKEN else None

def ship_file(path, dst_dir):
    # one record to dst_dir on the Hub: staged alone in its own folder, so the upload carries only it
    st = f'{OUT}/stage/{os.path.basename(path)[:-5]}'
    os.makedirs(st, exist_ok=True)
    shutil.copy(path, st)
    ship.submit(st, dst_dir)

def look_line(lk, kind):
    if not lk:
        return ''
    h7 = [b[7]['m'] for b in lk['blocks']]
    best = max(((b[h]['m'], i + 1, h) for i, b in enumerate(lk['blocks']) for h in range(len(b))))
    return (f" | the look: head 7 by block {' '.join(f'{x:.2f}' for x in h7)}; the largest {best[0]:.2f} (block {best[1]} head {best[2]}); "
            f"uniform {lk['uniform']:.3f}")

t_all, recs, n_read = time.time(), {}, 0
for i, (label, tag, kind, look_rows) in enumerate(CORES):
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP found: no new model')
        break
    name = f'{rr.room_name(tag)}.json'
    out = f'{OUT}/room/{name}'
    if f'{ROOM_DIR}/{name}' in _hub and not os.path.exists(out):
        recs[label] = json.load(open(hf_hub_download(REPO, f'{ROOM_DIR}/{name}', local_dir=f'{OUT}/hub', token=HF_TOKEN), encoding='utf-8'))
        print(f'{label}: its record is on the Hub')
        continue
    t0 = time.time()
    snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*', f'bed/reports/text/{rs.text_name(tag)}.json'], local_dir=f'{OUT}/w', token=HF_TOKEN)
    rec = rr.read(f'{OUT}/w/bed/weights/{tag}', out, HELD, kind, look_rows=look_rows, n_docs=TEXT_DOCS, mb=TEXT_MB, dev='cuda', log=lambda m: None)
    stored = json.load(open(f'{OUT}/w/bed/reports/text/{rs.text_name(tag)}.json', encoding='utf-8'))['bits']
    assert abs(rec['base']['bits'] - stored) < BASE_TOL, ('the read does not reproduce the stored text read', label, rec['base']['bits'], stored)
    recs[label] = rec
    n_read += 1
    if ship is not None:
        ship_file(out, ROOM_DIR)
    shutil.rmtree(f'{OUT}/w/bed/weights/{tag}', ignore_errors=True)
    costs = ' '.join(f"{k} {v['cost']:+.4f}" for k, v in rec['removed'].items())
    el = (time.time() - t_all) / 60
    print(f"{label}: own bits {rec['base']['bits']:.4f} (stored {stored:.4f}) | removed: {costs}{look_line(rec.get('look'), kind)} | "
          f"{time.time() - t0:.0f} s; {i + 1} of {len(CORES)} in {el:.1f} min, ~{el / (i + 1) * (len(CORES) - i - 1):.1f} min left", flush=True)
print(f'{n_read} models read here in {(time.time() - t_all) / 60:.1f} min (the plan: {PLAN_MIN:.0f} min for eighteen)')
if ship is not None:
    print('room records shipped' if ship.drain() else 'room records still retrying in the background')

In [ ]:
# C3 — THE TABLES: the channels' costs on the plain-rows models (each removal's held-out cost, all / at the repeated paths / elsewhere),
# the path models' head 7, and the look (head 7's mass on the continuation by block; the largest head's).
from IPython.display import Markdown, display
plain = [(l, r) for l, r in recs.items() if l.startswith('plain')]
if plain:
    names = list(plain[0][1]['removed'])
    rows = ['| model | own bits | ' + ' | '.join(names) + ' |', '|---' * (len(names) + 2) + '|']
    for l, r in plain:
        rows.append(f"| {l} | {r['base']['bits']:.4f} | " + ' | '.join(f"{r['removed'][n]['cost']:+.4f}" for n in names) + ' |')
    rows.append('| mean | - | ' + ' | '.join(f"{sum(r['removed'][n]['cost'] for _, r in plain) / len(plain):+.4f}" for n in names) + ' |')
    display(Markdown('**Each removal\'s held-out cost, bits a byte (the plain-rows models)**\n\n' + '\n'.join(rows)))
    rows = ['| model | ' + ' | '.join(f'{n}: repeated / elsewhere' for n in ('h7_anchors', 'h7_scores', 'h7_out')) + ' |', '|---' * 4 + '|']
    for l, r in plain:
        rows.append(f'| {l} | ' + ' | '.join(f"{r['removed'][n]['cost_repeat']:+.4f} / {r['removed'][n]['cost_other']:+.4f}" for n in ('h7_anchors', 'h7_scores', 'h7_out')) + ' |')
    display(Markdown('\n'.join(rows)))
path = [(l, r) for l, r in recs.items() if not l.startswith('plain')]
if path:
    rows = ['| model | own bits | head 7 anchor pairs removed: all / repeated / elsewhere | head 7 output removed: all / repeated / elsewhere |', '|---|---|---|---|']
    for l, r in path:
        a, o = r['removed']['h7_anchors'], r['removed']['h7_out']
        rows.append(f"| {l} | {r['base']['bits']:.4f} | {a['cost']:+.4f} / {a['cost_repeat']:+.4f} / {a['cost_other']:+.4f} | {o['cost']:+.4f} / {o['cost_repeat']:+.4f} / {o['cost_other']:+.4f} |")
    display(Markdown('**The path models\' head 7**\n\n' + '\n'.join(rows)))
looked = [(l, r) for l, r in recs.items() if r.get('look')]
if looked:
    nb = len(looked[0][1]['look']['blocks'])
    rows = ['| model | uniform | ' + ' | '.join(f'head 7, block {b + 1}' for b in range(nb)) + ' | the largest head (block, head): m, by length 1 / 2-3 / 4+ | its largest weight in the set |',
            '|---' * (nb + 4) + '|']
    for l, r in looked:
        lk = r['look']
        m, b, h = max(((lk['blocks'][i][j]['m'], i, j) for i in range(nb) for j in range(len(lk['blocks'][i]))))
        bl = lk['blocks'][b][h]['by_length']
        rows.append(f"| {l} | {lk['uniform']:.3f} | " + ' | '.join(f"{lk['blocks'][i][7]['m']:.3f}" for i in range(nb))
                    + f" | ({b + 1}, {h}): {m:.3f}, {' / '.join('-' if bl[k] is None else f'{bl[k]:.3f}' for k in ('1', '2-3', '4+'))} | {lk['blocks'][b][h]['argmax_share']:.3f} |")
    display(Markdown('**The look: the attention on the bytes after the earlier copies of the path, at the repeated positions**\n\n' + '\n'.join(rows)))